# Feed Forward Network with PyTorch

In this notebook, we will implement a neural network with the PyTorch llbrary, which provides us with an interface to easily create these machine learning models. We will use such implementation with the MNIST dataset created by Yann LeCun, Corinna Cortes, and Christopher J.C. Burge, to predict the real values of image representations of hand-drawn numbers.

We first begin our implementation by importing the necessary libraries.

In [79]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.datasets import mnist

We then import the MNIST dataset using the provided function by tensorflow.

We additinally divide our 60,000 training images into our training and validation subsets.

In [80]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_val, X_train = X_train[:10000], X_train[10000:]
y_val, y_train = y_train[:10000], y_train[10000:]

To improve the efficiency of our model, we will normalize the values of the pixels so that they stay within the 0-1 range.

In [81]:
X_val, X_train = X_val / .255, X_train / .255

The next step is to create our neural network using PyTorch.

We begin by defining our neural network class and its methods.

In [82]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, hidden3_size, hidden4_size, output_size):
        super().__init__()

        self.flatten = nn.Flatten()

        self.hidden1 = nn.Linear(input_size, hidden1_size)
        self.hidden2 = nn.Linear(hidden1_size, hidden2_size)
        self.hidden3 = nn.Linear(hidden2_size, hidden3_size)
        self.hidden4 = nn.Linear(hidden3_size, hidden4_size)

        self.output = nn.Linear(hidden4_size, output_size)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)

        x = self.hidden1(x)
        x = self.relu(x)

        x = self.hidden2(x)
        x = self.relu(x)

        x = self.hidden3(x)
        x = self.relu(x)

        x = self.hidden4(x)
        x = self.relu(x)
        
        x = self.output(x)
        return x
    


As observed by our class definition, our model is made out of 4 hidden layers. And an output layer, which serves as our classifier. We additionally include the Flatten method to transform our 28x28 bi-dimensional array input into a 1-dimensional array of 784 values.

Finally, we define our activation function to be the Rectified Linear Unit (ReLU).

We now define our optimizer, in this case we use Adam for its stableness.

We also need to create our model, we set the input size to 784 as this accepts the flattened images, we also define each hidden layer to be of sizes 80, and our final output layer size to be 10, with each output neuron corresponding to a number value from 0 to 9.

We also set up our device to be the GPU of the computer.

In [83]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FeedForwardNN(input_size=784, hidden1_size=80, hidden2_size=80, hidden3_size=80, hidden4_size=80, output_size=10)
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

As our loss function, we use Cross-Entropy, which is the correspding one for multi-class classification models.

This function already includes the final activation function SoftMax for the output layer, which saves us the manual implementation on our class.

In [84]:
loss_fn = nn.CrossEntropyLoss()

To facilitate learning and produce better results, we implement a dataloader to pass the data in batches to out neural network.

In [85]:
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

We also define our training function, this is where we compute the learning of our model, using backward propagation.

In [86]:
def train_batch(model, X_batch, y_batch, optimizer, loss_fn):
    predictions = model(X_batch)
    loss = loss_fn(predictions, y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss

We finally define our accuracy function, which will tell us how our model is performing so far on the validation subset. We also transform our validation data to tensors and set their device.

In [87]:
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)
X_val = X_val.detach().clone().float().to(device)
y_val = y_val.detach().clone().long().to(device)

def computeModelAccuracy(model, X, y):
    model.eval()

    with torch.no_grad():
        predictions = model(X)
        predicted = torch.argmax(predictions, dim=1)
        correct = (predicted == y).sum().item()
        total = y.size(0)

    return correct / total

Next step is to define our training loop.

Before that, we ensure to set our model to training mode.

As an important note, before calculating the current accuracy on the validation subset, we temporarily disable training mode during the function and set the model to evaluation mode to deactivate gradiant calculation.

In [88]:
model.train()

num_epochs = 30
for epoch in range(num_epochs):
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model, X_batch, y_batch, optimizer, loss_fn)

    accuracy = computeModelAccuracy(model, X_val, y_val)
    model.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation Accuracy: {accuracy:.2f}")

Epoch 1, Loss: 0.2972, Validation Accuracy: 0.94
Epoch 2, Loss: 0.0350, Validation Accuracy: 0.95
Epoch 3, Loss: 0.0346, Validation Accuracy: 0.96
Epoch 4, Loss: 0.1800, Validation Accuracy: 0.96
Epoch 5, Loss: 0.0132, Validation Accuracy: 0.96
Epoch 6, Loss: 0.0038, Validation Accuracy: 0.96
Epoch 7, Loss: 0.1573, Validation Accuracy: 0.95
Epoch 8, Loss: 0.1839, Validation Accuracy: 0.96
Epoch 9, Loss: 0.3173, Validation Accuracy: 0.96
Epoch 10, Loss: 0.0157, Validation Accuracy: 0.97
Epoch 11, Loss: 0.2093, Validation Accuracy: 0.96
Epoch 12, Loss: 0.0005, Validation Accuracy: 0.97
Epoch 13, Loss: 0.0113, Validation Accuracy: 0.96
Epoch 14, Loss: 0.4438, Validation Accuracy: 0.97
Epoch 15, Loss: 0.1907, Validation Accuracy: 0.97
Epoch 16, Loss: 0.0004, Validation Accuracy: 0.96
Epoch 17, Loss: 0.1439, Validation Accuracy: 0.96
Epoch 18, Loss: 0.0008, Validation Accuracy: 0.97
Epoch 19, Loss: 0.0263, Validation Accuracy: 0.96
Epoch 20, Loss: 0.0390, Validation Accuracy: 0.97
Epoch 21,

After training the model, we can observe its performance, having a final validation accuracy of 0.97.

## Additional models traning

In this part, we now create 3 additional neural networks to find the best performing one.

### Model 1

This model has 2 hidden layers of sizes 50 each, the SGD optimizer with a learning rate of 0.002, a mini batch size of 64, and a total of 20 epochs performed during training.

In [89]:
# Model definition
class FeedForwardNN_1(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, output_size):
        super().__init__()

        self.flatten = nn.Flatten()

        self.hidden1 = nn.Linear(input_size, hidden1_size)
        self.hidden2 = nn.Linear(hidden1_size, hidden2_size)

        self.output = nn.Linear(hidden2_size, output_size)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)

        x = self.hidden1(x)
        x = self.relu(x)

        x = self.hidden2(x)
        x = self.relu(x)
        
        x = self.output(x)
        return x
    
# Model creation
model1 = FeedForwardNN_1(input_size=784, hidden1_size=50, hidden2_size=50, output_size=10)
model1.to(device)

optimizer = optim.Adam(model1.parameters(), lr=0.002)

# Data preprocessing
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Model training
model1.train()

num_epochs = 20
for epoch in range(num_epochs):
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model1, X_batch, y_batch, optimizer, loss_fn)

    accuracy = computeModelAccuracy(model1, X_val, y_val)
    model1.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation Accuracy: {accuracy:.2f}")

Epoch 1, Loss: 0.9523, Validation Accuracy: 0.90
Epoch 2, Loss: 0.1241, Validation Accuracy: 0.93
Epoch 3, Loss: 0.6196, Validation Accuracy: 0.93
Epoch 4, Loss: 0.6092, Validation Accuracy: 0.93
Epoch 5, Loss: 0.7301, Validation Accuracy: 0.93
Epoch 6, Loss: 0.0088, Validation Accuracy: 0.94
Epoch 7, Loss: 1.1281, Validation Accuracy: 0.93
Epoch 8, Loss: 0.1907, Validation Accuracy: 0.94
Epoch 9, Loss: 0.1514, Validation Accuracy: 0.94
Epoch 10, Loss: 0.0013, Validation Accuracy: 0.94
Epoch 11, Loss: 0.1602, Validation Accuracy: 0.94
Epoch 12, Loss: 0.0252, Validation Accuracy: 0.94
Epoch 13, Loss: 0.8727, Validation Accuracy: 0.94
Epoch 14, Loss: 0.0826, Validation Accuracy: 0.94
Epoch 15, Loss: 0.0028, Validation Accuracy: 0.95
Epoch 16, Loss: 0.0248, Validation Accuracy: 0.94
Epoch 17, Loss: 0.1555, Validation Accuracy: 0.95
Epoch 18, Loss: 0.0234, Validation Accuracy: 0.94
Epoch 19, Loss: 0.0577, Validation Accuracy: 0.94
Epoch 20, Loss: 0.0256, Validation Accuracy: 0.95


After the training, the model capped at a 0.94 value of accuracy, which it achieved early on, offering a fast learning curve but without stellar performance

### Model 2

Next model has 5 hidden layers of sizes 80, 60, 70, 40, and 40 correspondingly, the Adam optimizer with a learning rate of 0.0015, a mini batch size of 32, and a total of 30 epochs performed during training.

In [90]:
# Model definition
class FeedForwardNN_2(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, hidden3_size, hidden4_size, hidden5_size, output_size):
        super().__init__()

        self.flatten = nn.Flatten()

        self.hidden1 = nn.Linear(input_size, hidden1_size)
        self.hidden2 = nn.Linear(hidden1_size, hidden2_size)
        self.hidden3 = nn.Linear(hidden2_size, hidden3_size)
        self.hidden4 = nn.Linear(hidden3_size, hidden4_size)
        self.hidden5 = nn.Linear(hidden4_size, hidden5_size)

        self.output = nn.Linear(hidden5_size, output_size)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)

        x = self.hidden1(x)
        x = self.relu(x)

        x = self.hidden2(x)
        x = self.relu(x)

        x = self.hidden3(x)
        x = self.relu(x)

        x = self.hidden4(x)
        x = self.relu(x)

        x = self.hidden5(x)
        x = self.relu(x)
        
        x = self.output(x)
        return x
    
# Model creation
model2 = FeedForwardNN_2(input_size=784, hidden1_size=80, hidden2_size=60, hidden3_size=70, hidden4_size=40, hidden5_size=40, output_size=10)
model2.to(device)

optimizer = optim.Adam(model2.parameters(), lr=0.0015)

# Data preprocessing
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Model training
model2.train()

num_epochs = 30
for epoch in range(num_epochs):
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model2, X_batch, y_batch, optimizer, loss_fn)

    accuracy = computeModelAccuracy(model2, X_val, y_val)
    model2.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation Accuracy: {accuracy:.2f}")

Epoch 1, Loss: 0.7579, Validation Accuracy: 0.94
Epoch 2, Loss: 0.0351, Validation Accuracy: 0.95
Epoch 3, Loss: 0.0213, Validation Accuracy: 0.95
Epoch 4, Loss: 0.0453, Validation Accuracy: 0.95
Epoch 5, Loss: 0.6343, Validation Accuracy: 0.96
Epoch 6, Loss: 0.0214, Validation Accuracy: 0.96
Epoch 7, Loss: 0.1529, Validation Accuracy: 0.96
Epoch 8, Loss: 0.0535, Validation Accuracy: 0.97
Epoch 9, Loss: 0.0033, Validation Accuracy: 0.96
Epoch 10, Loss: 0.2219, Validation Accuracy: 0.97
Epoch 11, Loss: 0.1670, Validation Accuracy: 0.96
Epoch 12, Loss: 0.0130, Validation Accuracy: 0.97
Epoch 13, Loss: 0.1489, Validation Accuracy: 0.96
Epoch 14, Loss: 0.0092, Validation Accuracy: 0.97
Epoch 15, Loss: 0.0043, Validation Accuracy: 0.97
Epoch 16, Loss: 0.2832, Validation Accuracy: 0.97
Epoch 17, Loss: 0.0012, Validation Accuracy: 0.97
Epoch 18, Loss: 0.1068, Validation Accuracy: 0.97
Epoch 19, Loss: 0.0013, Validation Accuracy: 0.97
Epoch 20, Loss: 0.0533, Validation Accuracy: 0.97
Epoch 21,

This model achieved a greater performance than the second one, with a validation accuracy of 0.97, which it achieved early on and making later epochs unnecessary.

### Model 3

The final new model has 3 hidden layers of sizes 80, 70, and 60, the Adam optimizer with a learning rate of 0.001, a mini batch size of 128, and a total of 20 epochs performed during training.

In [91]:
# Model definition
class FeedForwardNN_3(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, hidden3_size, output_size):
        super().__init__()

        self.flatten = nn.Flatten()

        self.hidden1 = nn.Linear(input_size, hidden1_size)
        self.hidden2 = nn.Linear(hidden1_size, hidden2_size)
        self.hidden3 = nn.Linear(hidden2_size, hidden3_size)

        self.output = nn.Linear(hidden3_size, output_size)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)

        x = self.hidden1(x)
        x = self.relu(x)

        x = self.hidden2(x)
        x = self.relu(x)

        x = self.hidden3(x)
        x = self.relu(x)
        
        x = self.output(x)
        return x
    
# Model creation
model3 = FeedForwardNN_3(input_size=784, hidden1_size=80, hidden2_size=70, hidden3_size=60, output_size=10)
model3.to(device)

optimizer = optim.Adam(model3.parameters(), lr=0.001)

# Data preprocessing
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# Model training
model3.train()

num_epochs = 20
for epoch in range(num_epochs):
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model3, X_batch, y_batch, optimizer, loss_fn)

    accuracy = computeModelAccuracy(model3, X_val, y_val)
    model3.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation Accuracy: {accuracy:.2f}")

Epoch 1, Loss: 0.4538, Validation Accuracy: 0.94
Epoch 2, Loss: 0.2777, Validation Accuracy: 0.95
Epoch 3, Loss: 0.1124, Validation Accuracy: 0.95
Epoch 4, Loss: 0.0897, Validation Accuracy: 0.96
Epoch 5, Loss: 0.0654, Validation Accuracy: 0.96
Epoch 6, Loss: 0.1848, Validation Accuracy: 0.96
Epoch 7, Loss: 0.1498, Validation Accuracy: 0.96
Epoch 8, Loss: 0.0698, Validation Accuracy: 0.96
Epoch 9, Loss: 0.0776, Validation Accuracy: 0.96
Epoch 10, Loss: 0.1119, Validation Accuracy: 0.96
Epoch 11, Loss: 0.1286, Validation Accuracy: 0.96
Epoch 12, Loss: 0.2017, Validation Accuracy: 0.97
Epoch 13, Loss: 0.0467, Validation Accuracy: 0.96
Epoch 14, Loss: 0.1272, Validation Accuracy: 0.97
Epoch 15, Loss: 0.0805, Validation Accuracy: 0.97
Epoch 16, Loss: 0.0175, Validation Accuracy: 0.97
Epoch 17, Loss: 0.0344, Validation Accuracy: 0.97
Epoch 18, Loss: 0.0418, Validation Accuracy: 0.97
Epoch 19, Loss: 0.0424, Validation Accuracy: 0.97
Epoch 20, Loss: 0.0328, Validation Accuracy: 0.96


This last model obtained a 0.97 validation accuracy similar to the other ones, but with less memory usage and a faster runtime due to mini batch size.

### Model evaluation

As the final part in this notebook, we evaluate the performance of our initial model, followed by the last 3 ones, to find out which one performs better under different hyperparameters.

In [94]:
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)
X_test = X_test.detach().clone().float().to(device)
y_test = y_test.detach().clone().long().to(device)

model.eval()

with torch.no_grad():
    # First model
    predictions = model(X_test)
    test_loss = loss_fn(predictions, y_test)

    accuracy = computeModelAccuracy(model, X_test, y_test)
    print(f"Model 1 - Loss: {test_loss.item():.4f}, Test Accuracy: {accuracy:.2f}")
    
    # Second model
    predictions = model1(X_test)
    test_loss = loss_fn(predictions, y_test)

    accuracy = computeModelAccuracy(model1, X_test, y_test)
    print(f"Model 2 - Loss: {test_loss.item():.4f}, Test Accuracy: {accuracy:.2f}")

    # Third model
    predictions = model2(X_test)
    test_loss = loss_fn(predictions, y_test)

    accuracy = computeModelAccuracy(model2, X_test, y_test)
    print(f"Model 3 - Loss: {test_loss.item():.4f}, Test Accuracy: {accuracy:.2f}")

    # Fourth model
    predictions = model3(X_test)
    test_loss = loss_fn(predictions, y_test)

    accuracy = computeModelAccuracy(model3, X_test, y_test)
    print(f"Model 4 - Loss: {test_loss.item():.4f}, Test Accuracy: {accuracy:.2f}")

Model 1 - Loss: 0.1714, Test Accuracy: 0.97
Model 2 - Loss: 0.4582, Test Accuracy: 0.94
Model 3 - Loss: 0.2288, Test Accuracy: 0.96
Model 4 - Loss: 0.2397, Test Accuracy: 0.96


C:\Users\carlo\AppData\Local\Temp\ipykernel_28888\2236872413.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)
C:\Users\carlo\AppData\Local\Temp\ipykernel_28888\2236872413.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test = torch.tensor(y_test, dtype=torch.long)


We can observe based on the loss value and accuracy on the test subset that the best performing model of them all was the first one, which consisted of 4 hidden layers of 80 neurons each. Additionally, the worst-performing one was the second one with few epochs, a bigger learning rate and only 2 hidden layers of 50 each.

In conclusion, this activity served as a great way to learn how to utilize PyTorch to program neural networks. Compared to a manual way, PyTorch offers a good toolset to create and adjust them really fast, which is essential in any project, especially when prototyping.

I already had past experience working with TensorFlow and creating the same exact neural network with its framework; however, I can now see why the deep learning community is leaning towards PyTorch, as it is more intuitive to use while also offering greater tweaking capabilities. Additionally, making the migration to a local environment compared to using Google Colab allowed me to use version control with Git correctly, which is great in these kinds of projects.

This practice also helped me get more familiar with neural network concepts and allowed me to gain more experience in Python.